<a href="https://colab.research.google.com/github/amina-mardiyyah/Data_Science_For_Life_Scientist_Course/blob/main/session2_model_evaluation_visualisation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <font color='blue'>Session 2 Practical: Deep Learning Model Evaluation & Visualisation</font>

**Course:** Data Science for Life Scientists  
**Dataset:** PubMed RCT 20k  
**Models:** Session 1 DNN baseline + Session 1 pretrained transformer

In this notebook we will:
- load the predictions saved from Session 1
- compare the DNN and transformer on the same test split
- inspect class-wise performance, confusion matrices, and error patterns
- visualise what each models get right and wrong
- Log the results to Weights & Biases (W&B)

The goal is to turn model outputs into evidence you can interpret and report.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Install dependencies

Only run the following cell if running in a Colab or a fresh environment.

In [ ]:
%%capture
!pip install uv
!uv pip install -U rich[jupyter]
!uv pip install wandb==0.17.9

In [ ]:
#Dependency imports
import json
import os
import random
from pathlib import Path

# Ignore unnecessary library warnings
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=SyntaxWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_fscore_support,
)

from IPython.display import display
from rich import print as rprint
from rich import inspect

import wandb


### <font color='red'>**N.b**</font>
*If running this notebook on a colab instead of local, mount your drive, then uncomment and run the following cell*

In [ ]:
#uncomment if running on colab
BASE_DIR = Path("/content/drive/MyDrive/Data_Science_For_Life_Scientist_26/Tutorials")

In [ ]:
# #uncomment if running on local
# BASE_DIR = Path.cwd()

In [ ]:
#constants
plt.style.use('seaborn-v0_8-whitegrid')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

LABEL_LIST = ['background', 'methods', 'results', 'objective', 'conclusions']
LABEL_TO_ID = {label: idx for idx, label in enumerate(LABEL_LIST)}
ID_TO_LABEL = {idx: label for label, idx in LABEL_TO_ID.items()}

ARTIFACT_DIR = BASE_DIR/ 'artifacts/session2_evaluation'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print('Label order:', LABEL_LIST)
print('Output folder:', ARTIFACT_DIR.resolve())

## Locate the Session 1 outputs

This notebook expects the predictions produced in Session 1:

- a **DNN** prediction table
- a **transformer** prediction table

If your files live in a different location, update the candidate paths below.

In [ ]:
# Paths to Session 1 outputs
# Update BASE_DIR above if your files are in a different folder.

DNN_OUTPUT_DIR = BASE_DIR / 'artifacts/session1A_dnn'
TRANSFORMER_OUTPUT_DIR = BASE_DIR / 'artifacts/session1B_transformer'

DNN_PRED_DIR = DNN_OUTPUT_DIR / 'predictions'
TRANSFORMER_PRED_DIR = TRANSFORMER_OUTPUT_DIR / 'predictions'

DNN_METRICS_PATH = DNN_OUTPUT_DIR / 'dnn_metrics_summary.json'
TRANSFORMER_METRICS_PATH = TRANSFORMER_OUTPUT_DIR / 'transformer_metrics_summary.json'



DNN_TEST_PRED_PATH = DNN_PRED_DIR / 'dnn_test_predictions.csv'


TRANSFORMER_TEST_PRED_PATH = TRANSFORMER_PRED_DIR / 'transformer_test_predictions.csv'


DNN_TEST_REPORT_PATH = DNN_PRED_DIR / 'dnn_test_classification_report.json'

TRANSFORMER_TEST_REPORT_PATH = TRANSFORMER_PRED_DIR / 'transformer_test_classification_report.json'



print('Loaded artifact paths:')
print('DNN metrics:', DNN_METRICS_PATH)
print('Transformer metrics:', TRANSFORMER_METRICS_PATH)
print('DNN test predictions:', DNN_TEST_PRED_PATH)
print('Transformer test predictions:', TRANSFORMER_TEST_PRED_PATH)
print('DNN test report:', DNN_TEST_REPORT_PATH)
print('Transformer test report:', TRANSFORMER_TEST_REPORT_PATH)


## Load and standardise the prediction tables

The helper functions below standardises our saved prediction tabels into a common schema:

- `example_id`
- `text`
- `true_label`
- `pred_label`
- `correct`
- `confidence`

In [ ]:
def coerce_labels(series):
    """Convert numeric label columns to readable class names."""
    if pd.api.types.is_numeric_dtype(series):
        return series.map(lambda x: ID_TO_LABEL[int(x)] if pd.notna(x) else x)

    # If values are numeric strings, map them too.
    sample = series.dropna().astype(str).head(10)
    if len(sample) and all(item.isdigit() for item in sample):
        return series.map(lambda x: ID_TO_LABEL[int(x)] if pd.notna(x) else x)

    return series.astype(str)


def load_prediction_table(path, model_name):
    """Load a saved prediction CSV and standardise column names."""
    path = Path(path)
    df = pd.read_csv(path).copy()
    df['example_id'] = np.arange(len(df))
    df['model'] = model_name

    # Standardise common column names.
    rename_map = {}
    if 'true_label_id' in df.columns and 'true_label' not in df.columns:
        rename_map['true_label_id'] = 'true_label'
    if 'pred_label_id' in df.columns and 'pred_label' not in df.columns:
        rename_map['pred_label_id'] = 'pred_label'
    df = df.rename(columns=rename_map)

    if 'true_label' not in df.columns or 'pred_label' not in df.columns:
        raise ValueError(f'Expected true_label and pred_label columns in {path}')

    df['true_label'] = coerce_labels(df['true_label'])
    df['pred_label'] = coerce_labels(df['pred_label'])

    if 'correct' not in df.columns:
        df['correct'] = df['true_label'] == df['pred_label']

    # The DNN model may not have confidence scores.
    if 'confidence' not in df.columns:
        df['confidence'] = np.nan

    keep_cols = [
        c for c in [
            'example_id', 'text', 'true_label', 'pred_label',
            'correct', 'confidence', 'model'
        ]
        if c in df.columns
    ]
    return df[keep_cols].copy()


In [ ]:
#helper function to load classification report
def load_json(path):
    """Load a JSON artifact from disk."""
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

In [ ]:
#load test prediction dataframes of each models
dnn_df = load_prediction_table(DNN_TEST_PRED_PATH, 'DNN')
transformer_df = load_prediction_table(TRANSFORMER_TEST_PRED_PATH, 'Transformer')

print('DNN test predictions shape:', dnn_df.shape)
print('Transformer test predictions shape:', transformer_df.shape)

dnn_df.head()


In [ ]:
transformer_df.head()

## Quick sanity check: class balance and examples

Before comparing scores, we can quickly review the label distribution of the test set and observe if it shares the same as the training set.

In [ ]:

true_label_counts = dnn_df['true_label'].value_counts().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 4.8))
sns.barplot(x=true_label_counts.index, y=true_label_counts.values, ax=ax, palette='viridis')
ax.set_title('Test-set label distribution')
ax.set_xlabel('Label')
ax.set_ylabel('Number of examples')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

print('Class distribution (%):')
percentage_distribution = (true_label_counts / true_label_counts.sum() * 100).round(1)
display(pd.DataFrame({'Label': percentage_distribution.index, 'Percentage': percentage_distribution.values}))

## Fetch computed overall metrics for both models

We will use the metrics we already computed from the previous sessions

In [ ]:
# Load compact metrics summaries saved by Session 1A and Session 1B.
dnn_metrics = load_json(DNN_METRICS_PATH)
transformer_metrics = load_json(TRANSFORMER_METRICS_PATH)

print('DNN summary keys:', dnn_metrics.keys())
print('Transformer summary keys:', transformer_metrics.keys())


In [ ]:
dnn_metrics

In [ ]:
transformer_metrics

In [ ]:
def summarise_metrics_from_json(summary, model_name=None, splits=('validation', 'test')):
    """Convert a saved compact metrics JSON into a tidy DataFrame.

    Expected summary structure:
    {
        'labels': [...],
        'validation': {'accuracy': ..., 'macro_f1': ..., ...},
        'test': {'accuracy': ..., 'macro_f1': ..., ...},
        'model_type': '...',
        ...
    }
    """
    rows = []
    for split in splits:
        if split not in summary:
            continue
        row = dict(summary[split])
        row['split'] = split
        row['model_name'] = model_name or summary.get('model_type', 'model')
        rows.append(row)
    return pd.DataFrame(rows)


dnn_metrics_df = summarise_metrics_from_json(dnn_metrics, model_name='DNN')
transformer_metrics_df = summarise_metrics_from_json(transformer_metrics, model_name='Transformer')

print('DNN metrics loaded from JSON:')
display(dnn_metrics_df.style.format(precision=4))

print('Transformer metrics loaded from JSON:')
display(transformer_metrics_df.style.format(precision=4))


In [ ]:
# Compare headline test metrics from the saved summaries.
metrics_df = pd.concat([dnn_metrics_df, transformer_metrics_df], ignore_index=True)

# Keep only the test split for the main model comparison table.
test_metrics_df = metrics_df.query("split == 'test'").copy()

preferred_cols = [
    'model_name', 'split', 'accuracy',
    'macro_precision', 'macro_recall', 'macro_f1',
    'weighted_precision', 'weighted_recall', 'weighted_f1',
]
existing_cols = [c for c in preferred_cols if c in test_metrics_df.columns]
test_metrics_df = test_metrics_df[existing_cols]

display(test_metrics_df.style.format(precision=4))

# Save overall metrics table
test_metrics_df.to_csv(ARTIFACT_DIR / 'overall_test_metrics.csv', index=False)


## Convert Classification report for each model to a compact dataframe

The classification report gives class-level precision, recall, F1, and support. We'll combine the report for both models into a single dataframe. This will make it easy for us to compare side-by-side class-level scores, differences, similarities etc

In [ ]:
dnn_test_report = load_json(DNN_TEST_REPORT_PATH)
transformer_test_report = load_json(TRANSFORMER_TEST_REPORT_PATH)

def report_to_dataframe(report, labels=LABEL_LIST):
    """Converts a saved sklearn classification_report dictionary into a DataFrame."""
    rows = []
    for label in labels:
        if label in report:
            rows.append({
                'label': label,
                'precision': report[label]['precision'],
                'recall': report[label]['recall'],
                'f1': report[label]['f1-score'],
                'support': report[label]['support'],
            })
    return pd.DataFrame(rows)


def display_saved_report(report, title):
    """Displays a saved classification report in a readable table format."""
    print(f'\n=== {title} ===')
    display(report_to_dataframe(report).style.format({
        'precision': '{:.4f}',
        'recall': '{:.4f}',
        'f1': '{:.4f}',
        'support': '{:.0f}',
    }))

    summary_rows = []
    for key in ['accuracy', 'macro avg', 'weighted avg']:
        if key in report:
            value = report[key]
            if isinstance(value, dict):
                summary_rows.append({
                    'summary': key,
                    'precision': value.get('precision', np.nan),
                    'recall': value.get('recall', np.nan),
                    'f1': value.get('f1-score', np.nan),
                    'support': value.get('support', np.nan),
                })
            else:
                summary_rows.append({
                    'summary': key,
                    'precision': np.nan,
                    'recall': np.nan,
                    'f1': value,
                    'support': np.nan,
                })
    if summary_rows:
        display(pd.DataFrame(summary_rows).style.format(precision=4))


In [ ]:
display_saved_report(dnn_test_report, 'DNN test classification report')

In [ ]:
display_saved_report(transformer_test_report, 'Transformer test classification report')

##### <font color='blue'>**Compare the models visually**</font>

We'll use a barchart to first visualise the overall metrics of each model side by side, then per-class performance

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
plot_df = metrics_df.set_index('model_name')[['accuracy', 'macro_f1', 'weighted_f1']]

# Melt the DataFrame to a long format suitable for seaborn
plot_df_melted = plot_df.reset_index().melt(id_vars='model_name', var_name='metric', value_name='score')

sns.barplot(x='model_name', y='score', hue='metric', data=plot_df_melted, ax=ax, palette='rocket')

# Annotate the bars with metric values
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', label_type='edge', padding=5)

ax.set_ylim(0, 1.1)
ax.set_title('DNN vs Transformer: overall performance on the test set')
ax.set_xlabel('Model Name')
ax.set_ylabel('Score')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Metric', bbox_to_anchor=(1.05, 1), loc='upper left', frameon=True)
plt.tight_layout()
plt.show()

#save comparison fig
fig.savefig(ARTIFACT_DIR / 'overall_metrics_comparison.png', dpi=200, bbox_inches='tight')

### <font color='blue'>**Per-class performance comparison**</font>

Overall scores can hide where each models behave differently or similarly. So we'll compare per-class metrics across all 5 classes for both models

In [ ]:
dnn_class = report_to_dataframe(dnn_test_report).assign(model='DNN')
transformer_class = report_to_dataframe(transformer_test_report).assign(model='Transformer')
class_df = pd.concat([dnn_class, transformer_class], ignore_index=True)

display(class_df.style.format({
    'precision': '{:.4f}',
    'recall': '{:.4f}',
    'f1': '{:.4f}',
    'support': '{:.0f}',
}))



In [ ]:
plot_df_metrics = class_df.melt(id_vars=['label', 'model'],
                                   value_vars=['precision', 'recall', 'f1'],
                                   var_name='metric', value_name='score')

# Define common plotting parameters
common_params = {
    'x': 'label',
    'y': 'score',
    'hue': 'model',
    'palette': 'rocket',
    'width': 0.9 #bar width
}

#precision
fig_precision, ax_precision = plt.subplots(figsize=(10, 6))
sns.barplot(data=plot_df_metrics[plot_df_metrics['metric'] == 'precision'],
            ax=ax_precision, **common_params)
ax_precision.set_title('Per-class Precision Comparison', fontsize=18)
ax_precision.set_xlabel('Label', fontsize=14)
ax_precision.set_ylabel('Precision Score', fontsize=14)
ax_precision.set_ylim(0, 1.1)
ax_precision.tick_params(axis='x', rotation=30, labelsize=12)
ax_precision.tick_params(axis='y', labelsize=12)

for container in ax_precision.containers:
    ax_precision.bar_label(container, fmt='%.2f', label_type='edge', padding=5, fontsize=11)
ax_precision.legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=12, title_fontsize=14)
plt.tight_layout()
plt.show()

#recall
fig_recall, ax_recall = plt.subplots(figsize=(10, 6))
sns.barplot(data=plot_df_metrics[plot_df_metrics['metric'] == 'recall'],
            ax=ax_recall, **common_params)
ax_recall.set_title('Per-class Recall Comparison', fontsize=18)
ax_recall.set_xlabel('Label', fontsize=14)
ax_recall.set_ylabel('Recall Score', fontsize=14)
ax_recall.set_ylim(0, 1.1)
ax_recall.tick_params(axis='x', rotation=30, labelsize=12)
ax_recall.tick_params(axis='y', labelsize=12)

for container in ax_recall.containers:
    ax_recall.bar_label(container, fmt='%.2f', label_type='edge', padding=5, fontsize=11)
ax_recall.legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=12, title_fontsize=14)
plt.tight_layout()
plt.show()

# f1
fig_f1, ax_f1 = plt.subplots(figsize=(10, 6))
sns.barplot(data=plot_df_metrics[plot_df_metrics['metric'] == 'f1'],
            ax=ax_f1, **common_params)
ax_f1.set_title('Per-class F1 Comparison', fontsize=18)
ax_f1.set_xlabel('Label', fontsize=14)
ax_f1.set_ylabel('F1 Score', fontsize=14)
ax_f1.set_ylim(0, 1.1)
ax_f1.tick_params(axis='x', rotation=30, labelsize=12)
ax_f1.tick_params(axis='y', labelsize=12)
for container in ax_f1.containers:
    ax_f1.bar_label(container, fmt='%.2f', label_type='edge', padding=5, fontsize=11)
ax_f1.legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=12, title_fontsize=14)
plt.tight_layout()
plt.show()

#save fig
fig_precision.savefig(ARTIFACT_DIR / 'model_per_class_precision_comparison.png', dpi=200, bbox_inches='tight')
fig_recall.savefig(ARTIFACT_DIR / 'model_per_call_recall_comparison.png', dpi=200, bbox_inches='tight')
fig_f1.savefig(ARTIFACT_DIR / 'model_per_class_f1_comparison.png', dpi=200, bbox_inches='tight')


### <font color='blue'>**Visualise Errors with Confusion matrices**</font>

We'll use the confusion matrix to inspect which labels are being confused by both models.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 9))

for ax, (model_name, df, cmap) in zip(
    axes,
    [
        ('DNN', dnn_df, 'Blues'),
        ('Transformer', transformer_df, 'Set3'),
    ],
):
    cm = confusion_matrix(df['true_label'], df['pred_label'], labels=LABEL_LIST)
    sns.heatmap(cm,
                annot=True,
                fmt='d',        # Format annotations as integers
                cmap=cmap,
                cbar=False,
                linewidths=0,
                xticklabels=LABEL_LIST,
                yticklabels=LABEL_LIST,
                ax=ax,
                annot_kws={'size': 14})
    ax.set_title(f'{model_name} Confusion Matrix', fontsize=14)
    ax.set_xlabel('Predicted Label', fontsize=14)
    ax.set_ylabel('True Label', fontsize=14)
    ax.tick_params(axis='x', rotation=45, labelsize=12)
    ax.tick_params(axis='y', rotation=0, labelsize=12)

plt.tight_layout()
plt.show()

From the above plot, we can already see some patterns in which labels are misclassified by both models. Now let's analyse which examples do the models agree or disagree on, and the sentences where they happen.

This is where apply some qualitative error analysis. We inspect examples that both models get right, examples that only one model gets right, and examples that both miss.

In [ ]:
def prepare_for_comparison(df, model_name):
    cols = ['example_id', 'text', 'true_label', 'pred_label', 'correct']
    if 'confidence' in df.columns:
        cols.append('confidence')
    out = df[cols].copy()
    suffix = f'_{model_name.lower()}'
    # Only rename columns that are specific to the model's predictions,
    # keep 'text' and 'true_label' as common columns for merging.
    rename = {c: f'{c}{suffix}' for c in out.columns if c not in ['example_id', 'text', 'true_label']}
    return out.rename(columns=rename)

comparison_df = prepare_for_comparison(dnn_df, 'DNN').merge(
    prepare_for_comparison(transformer_df, 'Transformer'),
    on=['example_id', 'text', 'true_label'],
    how='inner',
)

comparison_df['case'] = np.select(
    [
        comparison_df['correct_dnn'] & comparison_df['correct_transformer'],
        comparison_df['correct_dnn'] & ~comparison_df['correct_transformer'],
        ~comparison_df['correct_dnn'] & comparison_df['correct_transformer'],
    ],
    ['both correct', 'DNN only correct', 'Transformer only correct'],
    default='both wrong',
)

case_counts = comparison_df['case'].value_counts().reindex(['both correct', 'DNN only correct', 'Transformer only correct', 'both wrong'])
print('Agreement / disagreement counts:')
case_counts

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
case_counts.plot(kind='bar', ax=ax, color=['#54A24B', '#4C78A8', '#F58518', '#E45756'])

# Annotate the bars with their numeric counts
for container in ax.containers:
    ax.bar_label(container, fmt='%d', label_type='edge', padding=3)

ax.set_title('DNN vs Transformer: Agreement vs Disagreement analysis')
ax.set_ylabel('Number of examples')
ax.set_xlabel('Case')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

fig.savefig(ARTIFACT_DIR / 'agreement_disagreement_comparison.png', dpi=200, bbox_inches='tight')

## <font color='blue'>Inspect Each Model's mistakes</font>

We can look at a few misclassified examples from each model.

In [ ]:
#helper function
def show_mistakes(df, model_name, n=8):
    mistakes = df.loc[~df['correct']].copy()
    print(f'\n{model_name} misclassified examples: {len(mistakes):,}')
    if len(mistakes) == 0:
        print('No mistakes found.')
        return mistakes

    cols = ['text', 'true_label', 'pred_label']
    if 'confidence' in mistakes.columns:
        cols.append('confidence')
    display(mistakes[cols].head(n))
    return mistakes


In [ ]:
dnn_mistakes = show_mistakes(dnn_df, 'DNN')

In [ ]:
transformer_mistakes = show_mistakes(transformer_df, 'Transformer')

## <font color='blue'>Analyse Transformer's confidence: right vs wrong</font>

For confidence analysis, we'll stick to visualising just one model - <font color='red'>`the transformer model`</font>.
This will help us measure model's probability vs actual correctness.

In [ ]:

fig, ax = plt.subplots(figsize=(10, 8))
sns.violinplot(
    data=transformer_df,
    x='correct',
    y='confidence',
    hue='correct',
    palette='rocket',
    ax=ax,
    inner='quartile'
)
ax.set_title('Transformer confidence distribution by prediction correctness')
ax.set_xlabel('Prediction Correctness')
ax.set_ylabel('Max predicted probability')
ax.set_ylim(0, 1.05) #
plt.tight_layout()
plt.show()



In [ ]:
high_conf_errors = transformer_df.loc[~transformer_df['correct']].sort_values('confidence', ascending=False)
print('Most confident transformer mistakes DF :')
high_conf_errors[['text', 'true_label', 'pred_label', 'confidence']].head(10)

### <font color='blue'>**Extra Exercise**</font>
- Repeat the confidence analysis for the DNN model.
- Are the patterns the same? Does it display the same high confidence for misclassfied examples as the transformer model?
- Make a plot to show the classes with the highest confidence where the model was wrong for both models

## Save Session 2 outputs

We save the key comparison tables and figures used in this notebook. You can use these to also guide your project presentations, as a summary for the session, record-keeping etc.

In [ ]:
class_df

In [ ]:
comparison_df

In [ ]:
# Save a compact metrics summary that can be opened later in Session 2 or in a report
metrics_df.to_csv(ARTIFACT_DIR / 'overall_metrics.csv', index=False)
class_df.to_csv(ARTIFACT_DIR / 'per_class_metrics.csv', index=False)
comparison_df.to_csv(ARTIFACT_DIR / 'model_comparison_examples.csv', index=False)

## Optional: Logging results to Weights and Bias

This following cells are optional. If you have a W&B account and want to log the comparison tables and figures

In [ ]:
#use this cell to fetch your wandb token if working on colab
from google.colab import userdata
WANDB_TOKEN =  userdata.get('WANDB_TOKEN')


In [ ]:
#use this cell to fetch your wandb token if working locally
!uv pip install python-dotenv

from dotenv import load_dotenv
load_dotenv() #assuming keys are saved in a file `.env`
import os

WANDB_TOKEN =  os.environ.get('WANDB_TOKEN')


In [ ]:
#login to wandb
wandb.login(key=WANDB_TOKEN)

In [ ]:
with wandb.init(project='dsls_deep_learning_evaluation',
                entity='Mardiyyah',
                 name='session2_pubmed_rct20k_evaluation',
                 config={
        'dataset': 'PubMed RCT 20k',
        'models': ['DNN', 'Transformer'],
        'labels': LABEL_LIST,
    }) as run:


  run.log({
      'overall_metrics': wandb.Table(dataframe=metrics_df),
      'per_class_metrics': wandb.Table(dataframe=class_df),
      'confusion_matrices': wandb.Image(ARTIFACT_DIR / 'confusion_matrices.png'),
      'overall_comparison': wandb.Image(ARTIFACT_DIR / 'overall_metrics_comparison.png'),
  })



## Session Reflection questions

1. Which model performed better overall, and which metric supports that conclusion?

2. Which class is hardest to classify, and why might that be the case?

3. Does accuracy tell the full story for this dataset?

4. Which model makes more confident mistakes?

5. Which error patterns would you investigate first in a research setting?